# Niezgodnosci decision per planner

Notebook wczytuje plik results/new_dataset_results_merged.json i dla kazdego plannera wypisuje JSON z przypadkami, gdzie decision plannera nie jest rowne oczekiwanej decision dla query.

In [71]:
from pathlib import Path
import json

RESULTS_PATH = Path("results/risk_aware_results.json")

with RESULTS_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

records = data.get("results", [])

def extract_planner_payload(planner_entry):
    payload = planner_entry.get("result")
    if isinstance(payload, dict):
        return payload
    return {}

def extract_planner_decision(planner_entry):
    return extract_planner_payload(planner_entry).get("decision")

def decision_mismatches_for_planner(planner_name, category=None):
    mismatches = []
    for item in records:
        if category is not None and item.get("category") != category:
            continue

        expected_decision = item.get("decision")
        planner_entry = next((x for x in item.get("result", []) if x.get("planner") == planner_name), None)

        if planner_entry is None:
            continue

        payload = extract_planner_payload(planner_entry)
        planner_decision = payload.get("decision")
        if planner_decision != expected_decision:
            mismatches.append({
                "query": item.get("query"),
                "category": item.get("category"),
                "expected_decision": expected_decision,
                "planner_decision": planner_decision,
                "planner_answer": payload.get("answer")
            })

    return {
        "planner": planner_name,
        "category_filter": category,
        "mismatch_count": len(mismatches),
        "mismatches": mismatches
    }

def show_mismatches(planner_name, category=None):
    print(json.dumps(decision_mismatches_for_planner(planner_name, category=category), indent=2, ensure_ascii=False))

In [72]:
show_mismatches("SingleNearestGoalSemanticPlanner")

{
  "planner": "SingleNearestGoalSemanticPlanner",
  "category_filter": null,
  "mismatch_count": 0,
  "mismatches": []
}


In [73]:
show_mismatches("RuleBasedSequentialPlanner")

{
  "planner": "RuleBasedSequentialPlanner",
  "category_filter": null,
  "mismatch_count": 0,
  "mismatches": []
}


In [74]:
show_mismatches("SimpleLlmPlanner")

{
  "planner": "SimpleLlmPlanner",
  "category_filter": null,
  "mismatch_count": 0,
  "mismatches": []
}


In [75]:
show_mismatches("PlannerAgent")

{
  "planner": "PlannerAgent",
  "category_filter": null,
  "mismatch_count": 0,
  "mismatches": []
}


In [76]:
show_mismatches("PlannerAgent", category="multi_step")

{
  "planner": "PlannerAgent",
  "category_filter": "multi_step",
  "mismatch_count": 0,
  "mismatches": []
}


In [77]:
show_mismatches("RiskAwarePlannerAgent", category="multi_step")

{
  "planner": "RiskAwarePlannerAgent",
  "category_filter": "multi_step",
  "mismatch_count": 17,
  "mismatches": [
    {
      "query": "Before heading to the window in Bedroom 1, stop by the desk.",
      "category": "multi_step",
      "expected_decision": "execute",
      "planner_decision": "clarify",
      "planner_answer": "I need a bit more information before I can proceed. Order of actions is clear but 'the desk' is underspecified among multiple desks in the map; Bedroom 1 window and desk both exist, so no room-object conflict"
    },
    {
      "query": "Make your way to the armchair and then continue to the coffee table.",
      "category": "multi_step",
      "expected_decision": "execute",
      "planner_decision": "clarify",
      "planner_answer": "I need a bit more information before I can proceed. Multiple armchairs exist? No, only one armchair is present, but 'coffee table' is the standard object name while the map uses 'coffee_table_1'; Sequential navigation is cle